# 02. End to end demo on synthetic data

This notebook runs the complete pipeline on a synthetic panel so the moving parts can be inspected without downloading any external dataset. Cells run in order. Replace the synthetic panel with a real loader once the data is staged.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from stockml.data.preprocessing import drop_zero_volume, winsorize_returns
from stockml.features.pipeline import build_features
from stockml.labels.returns import multi_horizon_return_labels
from stockml.models import build_model
from stockml.training.trainer import train_walk_forward
from stockml.utils.io import project_root, read_yaml

feature_cfg = read_yaml(project_root() / "configs" / "features" / "standard_technicals.yaml")
splits_cfg = read_yaml(project_root() / "configs" / "splits" / "walk_forward.yaml")
model_cfg = read_yaml(project_root() / "configs" / "model" / "lightgbm.yaml")
splits_cfg["initial_train_years"] = 4
splits_cfg["validation_years"] = 1
splits_cfg["test_years"] = 1
splits_cfg["step_years"] = 1
model_cfg["params"]["num_iterations"] = 200
model_cfg["params"]["early_stopping_rounds"] = 20

In [ ]:
rng = np.random.default_rng(7)
dates = pd.bdate_range("2010-01-01", "2024-12-31")
panels = []
for ticker in ["AAA", "BBB", "CCC"]:
    rets = rng.normal(0.0006, 0.012, len(dates))
    close = 50.0 * np.exp(np.cumsum(rets))
    df = pd.DataFrame(
        {
            "open": close * (1 + rng.normal(0, 0.001, len(dates))),
            "high": close * (1 + np.abs(rng.normal(0, 0.005, len(dates)))),
            "low": close * (1 - np.abs(rng.normal(0, 0.005, len(dates)))),
            "close": close,
            "volume": rng.integers(1_000_000, 5_000_000, len(dates)),
            "ticker": ticker,
        },
        index=dates,
    )
    df.index.name = "date"
    panels.append(df)
panel = pd.concat(panels)
panel = drop_zero_volume(panel)
feats = build_features(panel, feature_cfg, market=None)
feats = multi_horizon_return_labels(feats, horizons=[1, 5, 20])
feats = winsorize_returns(feats, return_col="log_return_1")
feature_columns = [
    c
    for c in feats.columns
    if c
    not in {"open", "high", "low", "close", "volume", "ticker", "y_logret_h1", "y_logret_h5", "y_logret_h20"}
    and not c.startswith("log_return_")
]
len(feature_columns), feature_columns[:10]

In [ ]:
model = build_model(model_cfg, task="regression")
result = train_walk_forward(
    feats,
    feature_columns=feature_columns,
    label_column="y_logret_h5",
    model=model,
    task="regression",
    splits_cfg=splits_cfg,
)
result.aggregate()

In [ ]:
if result.feature_importances:
    pd.Series(result.feature_importances).sort_values(ascending=False).head(20)